# Multiverse — Stage 7 A/B Postmortem Diagnostics

閉じたDEV2000 Lineageの**診断専用**です。A_TOP10をA/Bだけで同一条件再生し、B実績・日次/週次/月次・一発依存度を出します。

**Segment C / ECON_HOLDOUT1000 は読みません。新ルール選抜もしません。**

操作は **ランタイム → すべてのセルを実行** だけです。


In [ ]:
from google.colab import drive
from pathlib import Path
import hashlib, json, shutil, subprocess, time

drive.mount('/content/drive')
MY=Path('/content/drive/MyDrive')
REPO=Path('/content/multiverse-ab-postmortem')
LOCAL=Path('/content/multiverse-ab-postmortem-input')
OUT=MY/'MULTIVERSE_ALL_MARKET_STAGE7_AB_POSTMORTEM_v1'
OUT.mkdir(parents=True, exist_ok=True)
OUT_JSON=OUT/'STAGE7_AB_POSTMORTEM_DIAGNOSTICS_v1.json'
LOG=OUT/'STAGE7_AB_POSTMORTEM_RUN_LOG_v1.txt'

def sha256_file(p):
    h=hashlib.sha256()
    with open(p,'rb') as f:
        for c in iter(lambda:f.read(1<<20),b''):
            h.update(c)
    return h.hexdigest()

def copy_exact(src,dst,expected):
    src=Path(src); dst=Path(dst); dst.parent.mkdir(parents=True,exist_ok=True)
    for n in range(1,4):
        try:
            if dst.exists(): dst.unlink()
            print(f'[COPY] {src.name} {n}/3')
            shutil.copyfile(src,dst)
            obs=sha256_file(dst)
            if obs!=expected: raise RuntimeError(f'SHA mismatch {obs} != {expected}')
            print(f'✅ COPY PASS {dst.name}')
            return dst
        except Exception as e:
            print('[RETRY]',repr(e))
            if dst.exists(): dst.unlink()
            if n==3: raise
            time.sleep(3)

existing=None
if OUT_JSON.is_file():
    try: existing=json.loads(OUT_JSON.read_text(encoding='utf-8'))
    except Exception: pass
if existing and existing.get('status')=='PASS_DIAGNOSTIC_ONLY' and existing.get('segment_C_access') is False and existing.get('ECON_HOLDOUT1000')=='SEALED':
    print('✅ A/B POSTMORTEM ALREADY PASS — NO RERUN')
else:
    if REPO.exists(): shutil.rmtree(REPO)
    subprocess.check_call(['git','clone','--depth','1','https://github.com/fufufu1116/multiverse-research.git',str(REPO)])
    checks={
      'v3/historical_all_market/stage7_ab_postmortem_diagnostics_v1.py':'a5747ec37324b456f8f76ecdba0a6324bf6cc576',
      'v3/historical_all_market/stage7_frozen_evaluator_v2.py':'ce8e109fa4c20f683ea1ec999b2fe5dd6f49c865',
      'v3/historical_all_market/stage456_preoutcome_decision_engine_v1.py':'a0ed6984969b0b98af1b074ef9fd2348f16604a0',
    }
    for rel,exp in checks.items():
        obs=subprocess.check_output(['git','-C',str(REPO),'hash-object',rel],text=True).strip()
        if obs!=exp: raise RuntimeError(f'FAIL-CLOSED blob mismatch {rel}: {obs} != {exp}')
    print('✅ EXACT DIAGNOSTIC CODE BINDINGS PASS')

    STAGE2=copy_exact(MY/'MULTIVERSE_ALL_MARKET_STAGE2_PRICE_EV_v1'/'DEV2000_ALL_MARKET_PRICE_EV_CATALOG_v1.jsonl',LOCAL/'stage2.jsonl','34ad32bed6e8b4d700864c46f4533bef1da254c7d87dc7ffe6ec266fd74530dc')
    PRED=copy_exact(MY/'MULTIVERSE_DEV2000_PREDICTION_LOCK_v3_IPHONE_LITE'/'DEV2000_CANDIDATE_A_B1A_RECONSTITUTED_v1_PREDICTIONS.csv',LOCAL/'pred.csv','772eca4d26f177b94a86ccf7c1b8486e3cdbac0cae454d76ce91fadeca5f1d51')
    UNIV=copy_exact(MY/'MULTIVERSE_DEV2000_UNIVERSE_RECOVERY'/'DEV2000_UNIVERSE_v1.csv',LOCAL/'universe.csv','eb561c9cad5121cf689b237d44a08d089f375a2b2b728e34e91a48338446f3b1')
    SETT=MY/'MULTIVERSE_ALL_MARKET_STAGE7_SETTLEMENT_EVAL_v1'/'SETTLEMENT_ONLY'
    ABREC=MY/'MULTIVERSE_ALL_MARKET_STAGE7_SETTLEMENT_EVAL_v1'/'STAGE7_AB_SELECTION_RECEIPT_v1.json'
    if not (SETT/'DEV2000_SETTLEMENT_A_v1.jsonl').is_file() or not (SETT/'DEV2000_SETTLEMENT_B_v1.jsonl').is_file() or not ABREC.is_file():
        raise RuntimeError('FAIL-CLOSED A/B settlement or receipt missing')

    script=REPO/'v3/historical_all_market/stage7_ab_postmortem_diagnostics_v1.py'
    cmd=['python',str(script),'--repo-root',str(REPO),'--stage2-jsonl',str(STAGE2),'--prediction-csv',str(PRED),'--universe-csv',str(UNIV),'--settlement-dir',str(SETT),'--ab-receipt',str(ABREC),'--out-json',str(OUT_JSON)]
    for n in range(1,4):
        p=subprocess.run(cmd,text=True,stdout=subprocess.PIPE,stderr=subprocess.STDOUT)
        LOG.write_text((LOG.read_text(encoding='utf-8') if LOG.is_file() else '')+f'\n=== attempt {n}/3 rc={p.returncode} ===\n'+p.stdout,encoding='utf-8')
        print('\n'.join(p.stdout.splitlines()[-40:]))
        if p.returncode==0 and OUT_JSON.is_file(): break
        if n==3: raise RuntimeError('FAIL-CLOSED A/B postmortem failed after 3 identical attempts')
        time.sleep(3)

x=json.loads(OUT_JSON.read_text(encoding='utf-8'))
if x.get('status')!='PASS_DIAGNOSTIC_ONLY' or x.get('segment_C_access') is not False or x.get('ECON_HOLDOUT1000')!='SEALED':
    raise RuntimeError('FAIL-CLOSED postmortem receipt state')
print('\n✅ STAGE7 A/B POSTMORTEM PASS')
for c in x['configs']:
    b=c['segments']['B']
    print(c['configuration_id'], {
      'B_ROI':b['overall']['realized_roi'],
      'B_bet_races':b['overall']['bet_race_count'],
      'B_hit_tickets':b['overall']['hit_ticket_count'],
      'weekly_positive_share':b['weekly']['positive_active_period_share'],
      'monthly_positive_share':b['monthly']['positive_active_period_share'],
      'largest_ticket_return_share':b['return_concentration']['largest_single_ticket_return_share'],
    })
